In [0]:
CREATE MATERIALIZED VIEW if not exists fintech_finpay.gold.dim_date as
SELECT 
    year(fecha)*10000+month(fecha)*100+day(fecha) as PK_Tiempo,
    fecha as date,
    year(fecha) as year,
    CASE 
        WHEN month(fecha) IN (1, 2, 3) THEN 'Q1'
        WHEN month(fecha) IN (4, 5, 6) THEN 'Q2'
        WHEN month(fecha) IN (7, 8, 9) THEN 'Q3'
        WHEN month(fecha) IN (10, 11, 12) THEN 'Q4'
    END as quarter,
    month(fecha) as month,
    CONCAT('S', weekofyear(fecha))  as week_of_year,
    day(fecha) as day
FROM (
    SELECT 
        explode(array_dates) as fecha
    FROM (
        SELECT 
            sequence(
                make_date(2018, 1, 1),
                make_date(2024, 12, 31),
                interval 1 day
            ) as array_dates
    )
);


CREATE MATERIALIZED VIEW if not exists fintech_finpay.gold.dim_user as
WITH transactions_per_type as (
    SELECT 
        user_id, transaction_type, COUNT(*) as transaction_count
    FROM fintech_finpay.gold.transactions
    WHERE status = 'aprobado'
    GROUP BY user_id, transaction_type
),
transactions_per_user as (
    SELECT 
        user_id, 
        sum(transaction_count) as total_transactions
    FROM transactions_per_type
    GROUP BY user_id
),
reversed_tranx_per_user as (
    SELECT
        t.user_id,
        CASE
            WHEN u.total_transactions > 0 THEN CAST(t.transaction_count/u.total_transactions as decimal(10,2))
            ELSE 0
        END as rate_reversed
    FROM transactions_per_type as t
    LEFT JOIN transactions_per_user as u
        ON t.user_id = u.user_id
    WHERE transaction_type = 'reversa'
)
SELECT 
    u.PK_Tiempo,
    u.user_id,
    u.full_name,
    u.document_id,
    u.email,
    u.phone,
    u.country,
    u.local_currency,
    u.registration_date,
    COALESCE(u.segment, 'desconocido') as segment,
    CASE
        WHEN COALESCE(t.rate_reversed, 0) >= 0.5 THEN 3
        WHEN COALESCE(t.rate_reversed, 0) >= 0.25 THEN 2
        WHEN COALESCE(t.rate_reversed, 0) >= 0.1 THEN 1
        ELSE 0
    END as user_risk_level
FROM fintech_finpay.gold.users as u
LEFT JOIN reversed_tranx_per_user as t
    ON u.user_id = t.user_id
;

CREATE MATERIALIZED VIEW if not exists fintech_finpay.gold.dim_merchant as
SELECT
    PK_Tiempo,
    merchant_id,
    merchant_name,
    category,
    country,
    local_currency,
    status,
    affiliation_date,
    CASE
        WHEN status <> 'inactivo' AND risk_level IS NULL AND date_diff( make_date(2024,01,02), affiliation_date) >= 360 THEN 'alto'
        WHEN status <> 'inactivo' AND risk_level IS NULL AND date_diff( make_date(2024,01,02), affiliation_date) >= 180 THEN 'medio'
        WHEN status <> 'inactivo' AND risk_level IS NULL AND date_diff( make_date(2024,01,02), affiliation_date) >= 90 THEN 'bajo'
        WHEN risk_level IS NULL THEN 'desconocido'
        ELSE risk_level
    END as risk_level
FROM fintech_finpay.gold.merchants
;

CREATE MATERIALIZED VIEW IF NOT EXISTS fintech_finpay.gold.dim_channel
WITH channels AS (
    SELECT DISTINCT channel
    FROM fintech_finpay.gold.transactions
)

SELECT 
    CONCAT('CH-', lpad(row_number() OVER(ORDER BY channel), 3, '0')) as channel_id,
    channel
FROM channels
;


CREATE MATERIALIZED VIEW IF NOT EXISTS fintech_finpay.gold.fact_transactions
WITH users as (
    SELECT 
        user_id,
        segment,
        local_currency,
        user_risk_level 
    FROM fintech_finpay.gold.dim_user
),
merchants as (
    SELECT
        merchant_id,
        local_currency,
        status,
        CASE
            WHEN risk_level = 'alto' THEN 3
            WHEN risk_level = 'medio' THEN 2
            WHEN risk_level = 'bajo' THEN 1
            ELSE 0
        END as merchant_risk_level
    FROM fintech_finpay.gold.dim_merchant
),
transactions_enhanced as (
    SELECT
        t.PK_Tiempo,
        t.transaction_id, 
        t.user_id,
        t.merchant_id,
        CASE
            WHEN channel = 'app' THEN 'CH-001'
            WHEN channel = 'pos' THEN 'CH-002'
            WHEN channel = 'web' THEN 'CH-003'
        END AS channel_id,
        CAST(CASE
            WHEN currency = 'PEN' THEN amount/3
            WHEN currency = 'ARS' THEN amount/2
            WHEN currency = 'MXN' THEN amount/5
            WHEN currency = 'COP' THEN amount/7
            WHEN currency = 'CLP' THEN amount/10
            ELSE amount
        END AS decimal(10,2)) as amount,
        t.transaction_type,
        CASE
            WHEN t.currency <> m.local_currency AND t.currency <> 'USD' 
                AND transaction_type = 'reversa' THEN 10*(merchant_risk_level + user_risk_level + 4) -- max 100
            WHEN t.currency <> m.local_currency 
                AND transaction_type = 'reversa' THEN 10*(merchant_risk_level + user_risk_level + 2) -- max 80
            WHEN t.currency <> m.local_currency 
                AND transaction_type <> 'reversa' THEN 5*(merchant_risk_level + user_risk_level + 4) -- max 50
            WHEN t.currency = m.local_currency 
                AND transaction_type = 'reversa' THEN 10*(merchant_risk_level + user_risk_level) -- max 60
            WHEN t.currency = m.local_currency 
                AND transaction_type <> 'reversa' THEN 5*(merchant_risk_level + user_risk_level) -- max 30
            ELSE 0
        END as risk_score
    --select *
    FROM fintech_finpay.gold.transactions as t
    LEFT JOIN users as u
        ON t.user_id = u.user_id
    LEFT JOIN merchants as m
        ON t.merchant_id = m.merchant_id
    WHERE t.status IN ('aprobado', 'rechazado')
        AND m.local_currency IS NOT NULL
        AND u.local_currency IS NOT NULL
)

SELECT
    PK_Tiempo,
    user_id,
    merchant_id,
    channel_id,
    sum(amount) as total_amount,
    avg(risk_score) as risk_score,
    count(
        CASE
            WHEN transaction_type = 'reversa' THEN 1
            ELSE NULL
        END
    ) as reversed_count,
    count(1) as transaction_count
FROM transactions_enhanced
GROUP BY 
    PK_Tiempo,
    user_id,
    merchant_id,
    channel_id
;